<a href="https://colab.research.google.com/github/manasesmutembei11/jupyter/blob/master/Copy_of_fcc_sms_text_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# import libraries
try:
    !pip install tf-nightly
except Exception:
    pass

import tensorflow as tf
import pandas as pd
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
import re
import string

print(tf.__version__)

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [ ]:
# Load dataset
train_df = pd.read_csv(train_file_path, sep="\t", names=["label", "message"])
test_df = pd.read_csv(test_file_path, sep="\t", names=["label", "message"])

# Convert labels to binary values
train_df["label"] = train_df["label"].map({"ham": 0, "spam": 1})
test_df["label"] = test_df["label"].map({"ham": 0, "spam": 1})

# Function to clean text
def clean_text(text):
    text = text.lower()
    text = re.sub(f"[{string.punctuation}]", "", text)  # Remove punctuation
    return text

# Apply text cleaning
train_df["message"] = train_df["message"].apply(clean_text)
test_df["message"] = test_df["message"].apply(clean_text)

# Tokenization
tokenizer = keras.preprocessing.text.Tokenizer(num_words=5000, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df["message"])

# Convert text to sequences
train_sequences = tokenizer.texts_to_sequences(train_df["message"])
test_sequences = tokenizer.texts_to_sequences(test_df["message"])

# Padding sequences
max_length = 100
train_padded = keras.preprocessing.sequence.pad_sequences(train_sequences, maxlen=max_length, padding="post")
test_padded = keras.preprocessing.sequence.pad_sequences(test_sequences, maxlen=max_length, padding="post")

# Labels
train_labels = train_df["label"].values
test_labels = test_df["label"].values

# Define Neural Network Model
model = keras.Sequential([
    keras.layers.Embedding(input_dim=5000, output_dim=16, input_length=max_length),
    keras.layers.GlobalAveragePooling1D(),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid")  # Binary classification
])

# Compile Model
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# Train Model
history = model.fit(train_padded, train_labels, epochs=100, validation_data=(test_padded, test_labels), verbose=1)

In [ ]:
# function to predict messages based on model
# (should return list containing prediction and label, ex. [0.008318834938108921, 'ham'])
def predict_message(pred_text):
    pred_text = clean_text(pred_text)  # Clean input text
    sequence = tokenizer.texts_to_sequences([pred_text])  # Tokenize
    padded = keras.preprocessing.sequence.pad_sequences(sequence, maxlen=max_length, padding="post")  # Pad
    prediction = model.predict(padded)[0][0]  # Get probability
    return [float(prediction), "spam" if prediction >= 0.5 else "ham"]  # Return classification

# Test function
pred_text = "how are you doing today?"
prediction = predict_message(pred_text)
print(prediction)

In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()
